# 09 KIỂM THỬ BẤT BIẾN THUỘC TÍNH BẰNG FUZZING (INVARIANT FUZZING TESTING)

Notebook này thực hiện kiểm định thuộc tính bất biến (Property-based / Invariant Fuzzing) qua hàng trăm nước đi ngẫu nhiên giữa các tác tử AI:
1. **Bất biến 1 (No Overlapping Pieces)**: Không bao giờ có 2 quân độc lập cùng tọa độ trên bàn cờ.
2. **Bất biến 2 (No Facing Commanders)**: Hai Tư lệnh không bao giờ lộ mặt nhìn nhau trên cùng hàng/cột mà không có quân cản.
3. **Bất biến 3 (Valid Terrain Domain)**: Không quân bộ nào rớt xuống biển sâu; Hải quân không đứng trên đất liền.
4. **Bất biến 4 (Two-Way Legality)**: Mọi action do `generate_legal_actions` sinh ra đều phải được `apply_action` chấp nhận (`result.accepted == True`).
5. **Bất biến 5 (Piece Count & Structural Integrity)**: Tổng số quân cờ luôn $\le 38$ và cấu trúc GameState luôn toàn vẹn.

In [1]:
from pathlib import Path
import sys
import time
import pandas as pd

# Cấu hình nạp module
cwd = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in [cwd, *cwd.parents] if (p / "packages").is_dir() and (p / "research").is_dir()), None)
if str(PROJECT_ROOT / "packages") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "packages"))

from commander_game_engine.board import Terrain
from commander_game_engine.game_state import GameState, PieceType, Player
from commander_game_engine.initial_setup import create_initial_state
from commander_game_engine.move_generator import generate_legal_actions
from commander_game_engine.engine import apply_action
from commander_game_engine import rules
from agents import RandomAgent

print("Nạp các module phục vụ Fuzzing Testing thành công!")

Nạp các module phục vụ Fuzzing Testing thành công!


## 1. Định nghĩa Hàm kiểm tra 5 Bất biến Cốt lõi (`verify_invariants`)

In [2]:
def verify_invariants(state: GameState, last_action, turn: int, game_idx: int):
    # 1. Bất biến không chồng lấn tọa độ
    occupied = {}
    for pid, piece in state.pieces.items():
        if piece.carried_by is None:
            assert piece.position is not None, f"Quân {pid} không có tọa độ!"
            assert piece.position not in occupied, f"Trùng tọa độ tại {piece.position} giữa {occupied[piece.position]} và {pid} ở ván {game_idx}, lượt {turn}!"
            occupied[piece.position] = pid

    # 2. Bất biến cấm Tư lệnh đối mặt
    vis_occ = rules._visible_occupancy(state)
    assert not rules.commanders_face(vis_occ), f"Hai Tư lệnh bị lộ mặt trực tiếp sau hành động {last_action} ở ván {game_idx}, lượt {turn}!"

    # 3. Bất biến địa hình
    for pid, piece in state.pieces.items():
        if piece.carried_by is None and piece.position is not None:
            region = state.board.get_region(*piece.position)
            if piece.piece_type == PieceType.NAVY:
                assert region in (Terrain.SEA, Terrain.COAST, Terrain.RIVER_BANK), f"Hải quân {pid} đứng ở đất liền: {piece.position} ({region})!"
            else:
                assert region != Terrain.SEA, f"Quân bộ {pid} đứng ở biển sâu: {piece.position}!"

    # 4. Bất biến số lượng quân cờ
    assert 0 < len(state.pieces) <= 38, f"Số lượng quân bất thường: {len(state.pieces)}"

print("Định nghĩa hàm verify_invariants hoàn tất và sẵn sàng kiểm thử.")

Định nghĩa hàm verify_invariants hoàn tất và sẵn sàng kiểm thử.


## 2. Thực thi Fuzzing qua 10 ván đấu ngẫu nhiên liên tiếp

In [3]:
NUM_GAMES = 10
MAX_TURNS_PER_GAME = 100

fuzz_stats = []
total_actions_checked = 0
total_invariants_checked = 0
t_start = time.perf_counter()

print(f"Bắt đầu Fuzzing {NUM_GAMES} ván cờ (tối đa {MAX_TURNS_PER_GAME} lượt/ván)...")

for g_idx in range(1, NUM_GAMES + 1):
    state = create_initial_state()
    bot_r = RandomAgent(name=f"Red_Bot_{g_idx}", seed=1000 + g_idx)
    bot_b = RandomAgent(name=f"Blue_Bot_{g_idx}", seed=2000 + g_idx)
    agents = {Player.RED: bot_r, Player.BLUE: bot_b}

    turns_played = 0
    outcome = "Max turns reached"

    for turn in range(1, MAX_TURNS_PER_GAME + 1):
        player = state.current_player
        legal_actions = generate_legal_actions(state)
        if not legal_actions:
            outcome = f"Phe {player.value.upper()} hết nước đi"
            turns_played = turn
            break

        decision = agents[player].select_action(state, legal_actions)
        act = decision.action
        total_actions_checked += 1

        # Bất biến 4: Engine PHẢI duyệt nước đi do generator sinh ra
        result = apply_action(state, act)
        assert result.accepted == True, f"Engine từ chối nước đi hợp lệ {act} ở ván {g_idx}, lượt {turn}: {result.verdict.reason}"

        # Kiểm tra toàn bộ 5 bất biến trên trạng thái mới
        verify_invariants(result.state, act, turn, g_idx)
        total_invariants_checked += 5
        turns_played = turn
        state = result.state

        if result.terminal:
            winner = result.winner.value.upper() if result.winner else "None"
            outcome = f"Phe {winner} thắng (Diệt Tư lệnh)"
            break

    fuzz_stats.append({
        "Ván": g_idx,
        "Số lượt": turns_played,
        "Kết quả": outcome,
        "Số quân còn lại": len(state.pieces)
    })

fuzz_time = time.perf_counter() - t_start
print(f"Fuzzing hoàn tất trong {fuzz_time:.2f} giây!")

Bắt đầu Fuzzing 10 ván cờ (tối đa 100 lượt/ván)...
Fuzzing hoàn tất trong 9.92 giây!


## 3. Bảng tổng kết Fuzzing & Phân tích Đồ thị Bất biến

In [4]:
df_fuzz = pd.DataFrame(fuzz_stats)
display(df_fuzz)

print(f"\n--- TỔNG KẾT BẤT BIẾN (INVARIANT DASHBOARD) ---")
print(f"Tổng số ván đấu mô phỏng: {NUM_GAMES} ván")
print(f"Tổng số nước đi đã duyệt: {total_actions_checked} nước đi")
print(f"Tổng số lượt kiểm tra bất biến: {total_invariants_checked} lần kiểm tra")
print(f"Số lỗi vi phạm phát hiện: 0 VI PHẠM (100% PASS!)")
print(f"Tốc độ kiểm tra bất biến: {total_actions_checked / fuzz_time:.1f} transitions / giây")

assert total_actions_checked > 200, "Phải kiểm tra ít nhất 200 bước chuyển trạng thái!"
print("\nPASS: Toàn bộ 5 bất biến cốt lõi được giữ vững tuyệt đối trong suốt quá trình vận hành!")

   Ván  Số lượt            Kết quả  Số quân còn lại
0    1      100  Max turns reached               32
1    2      100  Max turns reached               36
2    3      100  Max turns reached               36
3    4      100  Max turns reached               38
4    5      100  Max turns reached               34
5    6      100  Max turns reached               34
6    7      100  Max turns reached               36
7    8      100  Max turns reached               34
8    9      100  Max turns reached               32
9   10      100  Max turns reached               36

--- TỔNG KẾT BẤT BIẾN (INVARIANT DASHBOARD) ---
Tổng số ván đấu mô phỏng: 10 ván
Tổng số nước đi đã duyệt: 1000 nước đi
Tổng số lượt kiểm tra bất biến: 5000 lần kiểm tra
Số lỗi vi phạm phát hiện: 0 VI PHẠM (100% PASS!)
Tốc độ kiểm tra bất biến: 100.8 transitions / giây

PASS: Toàn bộ 5 bất biến cốt lõi được giữ vững tuyệt đối trong suốt quá trình vận hành!
